# 08 · Importancia de variables y saliencia
Responde a: *¿la red aprendió que ROTI antecede al S4, o es una asunción?* Dos métodos complementarios sobre **Validación**: importancia por permutación (global) y gradientes integrados (variable × minuto de la ventana) en ventanas de **inicio** de evento.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos, correr_variante
from s4.train import configurar_gpu
from s4 import features as ft, explain, stats, evaluate as ev
configurar_gpu()
cfg = cargar_config('../config/ablacion_2025_2026.yaml')
datos = preparar_datos(cfg)
filas, modelos, parts = correr_variante(cfg, datos, ft.CONJUNTOS["E_completo"], "E explicabilidad", "explicabilidad",
                                        semillas=[42], guardar_modelo=True)
modelo, esc, p = modelos[0][1], parts["escalador"], parts["val"]

In [ ]:
imp_glob = explain.importancia_permutacion(modelo, p, esc, stats.m_rmse_global)
imp_ev = explain.importancia_permutacion(modelo, p, esc, stats.m_rmse_evento(), solo_eventos=True)
imp = imp_glob.merge(imp_ev, on="variable", suffixes=("_global", "_eventos"))
imp.to_csv(ruta(cfg,"reportes")/"tables/tabla_importancia_permutacion.csv", index=False); imp.round(5)

In [ ]:
fase = ev.fase_ventana(p.ultimo_real, p.y_real)
idx = np.flatnonzero(fase == "inicio")[:300]
atrib = explain.gradientes_integrados(modelo, p.X_lote(idx))
display(explain.resumen_saliencia(atrib, p.features))
mapa = np.abs(atrib).mean(axis=0).T            # (variables, minutos)
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(mapa, aspect="auto", cmap="magma")
ax.set_yticks(range(len(p.features))); ax.set_yticklabels(p.features)
ax.set_xlabel(f"minuto de la ventana (0 = t-{p.lookback-1}, {p.lookback-1} = t)")
plt.colorbar(im); plt.title("Gradientes integrados en ventanas de inicio de evento")
plt.tight_layout(); plt.savefig(ruta(cfg,"reportes")/"figures/fig_saliencia_inicio.png", dpi=200); plt.show()